## Exercise 1 – CUDA Environment Setup and Hello GPU

In [4]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

Writing hello.cu


In [5]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


In [6]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello | tee hello_output.txt

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


In [7]:
!cat hello_output.txt

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


Exercise 2 – Simple **Calculation**: Adding Two Numbers
---




In [8]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (the NVIDIA GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;
    int *d_a, *d_b, *d_c;
    int size = sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = 2;
    b = 7;

    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    add<<<1, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);

    printf("Result is %d\n", c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex2_add.cu


In [9]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add | tee ex2_output.txt

Result is 9


In [10]:
!cat ex2_output.txt

Result is 9


# Exercise 3 – Vector Addition Using 512 Blocks and One Thread

In [11]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // Each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and create random values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy input arrays from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 512 blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print all 512 results
    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Verify GPU answers
    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex3_blocks.cu


In [12]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks | tee ex3_output.txt

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
35 + 57 = 92
86 + 14 = 100
92 + 68 = 160
49 + 5 = 54
21 + 58 = 79
62 + 12 = 74
27 + 86 = 113
90 + 0 = 90
59 + 46 = 105
63 + 26 = 89
26 + 94 = 120
40 + 16 = 56
26 + 52 = 78
72 + 78 = 150
36 + 29 = 65
11 + 46 = 57
68 + 90 = 158
67 + 47 = 114
29 + 70 = 99
82 + 51 = 133
30 + 80 = 110
62 + 31 = 93
23 + 93 = 116
67 + 57 = 124
35 + 27 = 62
29 + 12 = 41
2 + 86 = 88
22 + 14 = 36
58 + 55 = 113
69 + 12 = 81
67 + 90 = 157
93 + 12 = 105
56 + 79 = 135
11 + 10 = 21
42 + 69 = 111
29 + 89 = 118
73 + 74 = 147
21 + 55 = 76
19 + 41 = 60
84 + 20 = 104
37 + 33 = 70
98 + 87 = 185
24 + 88 = 112
15 + 38 = 53
70 + 66 = 136
13 + 70 = 83
26 + 84 = 110
91 + 56 = 147
80 + 17 = 97
56 + 6 = 62
73 + 60 = 133
62 + 49 = 111
70 + 37 = 107
96 + 5 = 101
81 + 59 = 140
5 + 17 = 22
25 + 18 = 43
84 + 45 = 129
27 + 83 = 110
36 + 73 = 109
5 + 58 = 63
46 + 73 = 119
29 + 37 = 66
13 + 89 = 102
57 + 83 = 140
24 + 7 = 31
95 + 78 = 173
82 + 57 = 139
45 + 14 = 59
14 +

In [13]:
!tail -n 10 ex3_output.txt

17 + 50 = 67
23 + 62 = 85
61 + 82 = 143
81 + 99 = 180
9 + 17 = 26
90 + 54 = 144
25 + 73 = 98
96 + 15 = 111
67 + 6 = 73
Verification: 0 mismatches out of 512


#  Exercise 4 – Vector Addition Using One Block and 512 Threads

In [15]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 1 block, 512 threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex4_threads.cu


In [16]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads | tee ex4_output.txt

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

In [17]:
!tail -n 10 ex4_output.txt

503) 17 + 50 = 67
504) 23 + 62 = 85
505) 61 + 82 = 143
506) 81 + 99 = 180
507) 9 + 17 = 26
508) 90 + 54 = 144
509) 25 + 73 = 98
510) 96 + 15 = 111
511) 67 + 6 = 73
Verification: 0 mismatches out of 512


### Experiment – Find the Thread Limit (N = 2048)

In [21]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 1 block, 512 threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex4_threads.cu


In [19]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads 2>&1 | tee ex4_2048_failure_output.txt

Kernel launch failed: invalid argument
0) 83 + 51 = 0
1) 86 + 80 = 0
2) 77 + 20 = 0
3) 15 + 97 = 0
4) 93 + 0 = 0
5) 35 + 88 = 0
6) 86 + 61 = 0
7) 92 + 96 = 0
8) 49 + 74 = 0
9) 21 + 1 = 0
10) 62 + 14 = 0
11) 27 + 69 = 0
12) 90 + 28 = 0
13) 59 + 16 = 0
14) 63 + 52 = 0
15) 26 + 82 = 0
16) 40 + 25 = 0
17) 26 + 82 = 0
18) 72 + 33 = 0
19) 36 + 2 = 0
20) 11 + 77 = 0
21) 68 + 23 = 0
22) 67 + 49 = 0
23) 29 + 90 = 0
24) 82 + 51 = 0
25) 30 + 35 = 0
26) 62 + 60 = 0
27) 23 + 46 = 0
28) 67 + 99 = 0
29) 35 + 47 = 0
30) 29 + 29 = 0
31) 2 + 2 = 0
32) 22 + 28 = 0
33) 58 + 49 = 0
34) 69 + 99 = 0
35) 67 + 28 = 0
36) 93 + 89 = 0
37) 56 + 13 = 0
38) 11 + 76 = 0
39) 42 + 63 = 0
40) 29 + 66 = 0
41) 73 + 90 = 0
42) 21 + 84 = 0
43) 19 + 94 = 0
44) 84 + 59 = 0
45) 37 + 36 = 0
46) 98 + 76 = 0
47) 24 + 84 = 0
48) 15 + 71 = 0
49) 70 + 9 = 0
50) 13 + 38 = 0
51) 26 + 0 = 0
52) 91 + 84 = 0
53) 80 + 39 = 0
54) 56 + 90 = 0
55) 73 + 35 = 0
56) 62 + 75 = 0
57) 70 + 50 = 0
58) 96 + 81 = 0
59) 81 + 26 = 0
60) 5 + 50 = 0
61)

In [20]:
!grep "Kernel launch failed" ex4_2048_failure_output.txt

Kernel launch failed: invalid argument


In [22]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads | tee ex4_output.txt

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

In [23]:
!tail -n 3 ex4_output.txt

510) 96 + 15 = 111
511) 67 + 6 = 73
Verification: 0 mismatches out of 512


#  Exercise 5 – Multiply Two Vectors Using Blocks and Threads

In [24]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks =
        (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex5_vecmul.cu


In [25]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul | tee ex5_output.txt

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999001) 76 x 74 = 5624
9999002) 41 x 94 = 3854
9999003) 8 x 0 = 0
9999004) 1 x 41 = 41
9999005) 28 x 52 = 1456
9999006) 45 x 4 = 180
9999007) 63 x 35 = 2205
9999008) 89 x 97 = 8633
9999009) 10 x 71 = 710
9999010) 74 x 91 = 6734
9999011) 65 x 41 = 2665
9999012) 29 x 60 = 1740
9999013) 53 x 5 = 265
9999014) 35 x 88 = 3080
9999015) 18 x 12 = 216
9999016) 17 x 12 = 204
9999017) 8 x 65 = 520
9999018) 51 x 34 = 1734
9999019) 79 x 64 = 5056
9999020) 13 x 89 = 1157
9999021) 10 x 85 = 850
9999022) 7 x 2 = 14
9999023) 31 x 51 = 1581
9999024) 88 x 86 = 7568
9999025) 59 x 8 = 472
9999026) 94 x 50 = 4700
9999027) 58 x 94 = 5452
9999028) 5 x 53 = 265
9999029) 36 x 67 = 2412
9999030) 80 x 45 = 3600
9999031) 33 x 5 = 165
9999032) 64 x 41 = 2624
9999033) 21 x 91 = 1911
9999034) 93 x 6 = 558
9999035) 65 x 34 = 2210
9999036) 1 x 43 = 43
9999037) 90 x 62 = 5580
9999038) 80 x 21 = 1680
9999039) 90 x 92 = 8280
9999040) 1 x 86 = 86
9999041) 54 x 1

In [26]:
!tail -n 10 ex5_output.txt

9999991) 95 x 13 = 1235
9999992) 68 x 50 = 3400
9999993) 84 x 81 = 6804
9999994) 24 x 52 = 1248
9999995) 74 x 97 = 7178
9999996) 40 x 38 = 1520
9999997) 24 x 63 = 1512
9999998) 7 x 69 = 483
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000


In [27]:
!head -n 2 ex5_output.txt

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952


# Exercise 6 – 2D Matrix Element-wise Multiplication

In [31]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]  (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);

    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    mulElem<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, ROWS, COLS
    );

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (size_t r = count - 1000; r < count; r++) {
        printf("C[%zu][%zu] = %d x %d = %d\n",
               r / COLS,
               r % COLS,
               a[r],
               b[r],
               c[r]);
    }

    size_t errors = 0;

    for (size_t r = 0; r < count; r++) {
        if (c[r] != a[r] * b[r])
            errors++;
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex6_matrix.cu


In [32]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix | tee ex6_output.txt

Grid: 313 x 625 blocks
C[9999][9000] = 29 x 56 = 1624
C[9999][9001] = 58 x 69 = 4002
C[9999][9002] = 76 x 50 = 3800
C[9999][9003] = 68 x 21 = 1428
C[9999][9004] = 67 x 78 = 5226
C[9999][9005] = 98 x 11 = 1078
C[9999][9006] = 3 x 29 = 87
C[9999][9007] = 69 x 52 = 3588
C[9999][9008] = 93 x 64 = 5952
C[9999][9009] = 60 x 65 = 3900
C[9999][9010] = 49 x 77 = 3773
C[9999][9011] = 5 x 44 = 220
C[9999][9012] = 6 x 77 = 462
C[9999][9013] = 17 x 31 = 527
C[9999][9014] = 45 x 65 = 2925
C[9999][9015] = 17 x 50 = 850
C[9999][9016] = 59 x 2 = 118
C[9999][9017] = 68 x 4 = 272
C[9999][9018] = 70 x 28 = 1960
C[9999][9019] = 54 x 11 = 594
C[9999][9020] = 7 x 87 = 609
C[9999][9021] = 29 x 93 = 2697
C[9999][9022] = 12 x 88 = 1056
C[9999][9023] = 14 x 26 = 364
C[9999][9024] = 74 x 6 = 444
C[9999][9025] = 24 x 52 = 1248
C[9999][9026] = 4 x 14 = 56
C[9999][9027] = 75 x 76 = 5700
C[9999][9028] = 0 x 40 = 0
C[9999][9029] = 65 x 7 = 455
C[9999][9030] = 67 x 13 = 871
C[9999][9031] = 29 x 96 = 2784
C[9999][9032] 

In [33]:
!head -n 2 ex6_output.txt

Grid: 313 x 625 blocks
C[9999][9000] = 29 x 56 = 1624


In [34]:
!tail -n 10 ex6_output.txt

C[9999][9991] = 77 x 0 = 0
C[9999][9992] = 36 x 87 = 3132
C[9999][9993] = 57 x 62 = 3534
C[9999][9994] = 95 x 27 = 2565
C[9999][9995] = 94 x 22 = 2068
C[9999][9996] = 14 x 5 = 70
C[9999][9997] = 69 x 82 = 5658
C[9999][9998] = 9 x 37 = 333
C[9999][9999] = 86 x 21 = 1806
Verification: 0 mismatches out of 100000000


In [35]:
!ls -lh *.cu *.txt

-rw-r--r-- 1 root root  848 Oct  7 11:14 ex2_add.cu
-rw-r--r-- 1 root root   12 Oct  7 11:14 ex2_output.txt
-rw-r--r-- 1 root root 1.6K Oct  7 11:21 ex3_blocks.cu
-rw-r--r-- 1 root root 6.8K Oct  7 11:21 ex3_output.txt
-rw-r--r-- 1 root root  35K Oct  7 11:39 ex4_2048_failure_output.txt
-rw-r--r-- 1 root root 9.2K Oct  7 11:45 ex4_output.txt
-rw-r--r-- 1 root root 1.4K Oct  7 11:43 ex4_threads.cu
-rw-r--r-- 1 root root  23K Oct  7 11:51 ex5_output.txt
-rw-r--r-- 1 root root 1.7K Oct  7 11:49 ex5_vecmul.cu
-rw-r--r-- 1 root root 2.2K Oct  7 11:58 ex6_matrix.cu
-rw-r--r-- 1 root root  30K Oct  7 11:58 ex6_output.txt
-rw-r--r-- 1 root root  188 Oct  7 11:05 hello.cu
-rw-r--r-- 1 root root   96 Oct  7 11:09 hello_output.txt


In [36]:
!zip SE3082_Lab09_IT24101673_Files.zip *.cu *.txt

  adding: ex2_add.cu (deflated 53%)
  adding: ex3_blocks.cu (deflated 58%)
  adding: ex4_threads.cu (deflated 59%)
  adding: ex5_vecmul.cu (deflated 61%)
  adding: ex6_matrix.cu (deflated 62%)
  adding: hello.cu (deflated 20%)
  adding: ex2_output.txt (stored 0%)
  adding: ex3_output.txt (deflated 64%)
  adding: ex4_2048_failure_output.txt (deflated 69%)
  adding: ex4_output.txt (deflated 61%)
  adding: ex5_output.txt (deflated 67%)
  adding: ex6_output.txt (deflated 73%)
  adding: hello_output.txt (deflated 61%)
